## Entendimiento del problema

### Carlos Javier Palacios Sanchez

### 2026-08-15

### Description:

Descargar los datos y entender el problema a realizar, contestar en el notebook

1. ¿Cual es el objetivo del problema?

2. ¿Cómo se usará su solución?

3. ¿Cuáles son las soluciones actuales (si las hay)?

4. ¿Cómo se debe enmarcar este problema (supervisado / no supervisado, en línea / fuera de línea, etc.)

5. ¿Cómo se debe medir el desempeño o el rendimiento de la solución, una primera intuicion?

6. ¿La medida de desempeño está alineada con el objetivo del problema?

7. ¿Cuál sería el desempeño o rendimiento mínimo necesario para alcanzar el objetivo del problema?

8. ¿Cuáles son los problemas parecidos? ¿Se puede reutilizar experiencias o herramientas ya creadas?

9. ¿Hay experiencia del problema disponible?

10. ¿Cómo se puede resolver el problema manualmente?

11. Hacer un listado de los supuestos que hay hasta este momento.

12. Cual es la fuente de los datos?

13. Como se actualizan los datos?

14. Cada cuanto tiempo se actualizan los datos


## 📊 Answers

¿Cuál es el objetivo del problema?
Predecir si un paciente tiene enfermedad cardiaca ( disease : 1 = sí, 0 = no) a partir de 13 variables clínicas de rutina.
El fin es apoyar la priorización: detectar temprano a quienes deben ir a angiografía y evitar procedimientos invasivos
innecesarios.

¿Cómo se usará su solución?
Como soporte a la decisión clínica, no como diagnóstico automático. El médico ingresa los datos del paciente, el
modelo devuelve una probabilidad de riesgo, y esa probabilidad alimenta la decisión de remitir a estudios
confirmatorios. La decisión final siempre es del médico.

¿Cuáles son las soluciones actuales (si las hay)?
El juicio clínico del cardiólogo, las escalas de riesgo validadas (Framingham, SCORE2, ACC/AHA) y la angiografía
coronaria como estándar de oro. La angiografía es confiable pero invasiva y costosa, por lo que no sirve para tamizaje
masivo.

¿Cómo se debe enmarcar este problema?
Aprendizaje supervisado, clasificación binaria, fuera de línea (batch) y basado en modelo. Los datos son
estáticos e históricos, no hay flujo continuo de registros.

¿Cómo se debe medir el desempeño? Una primera intuición
La exactitud no basta: los errores tienen costos muy distintos. Un falso negativo (enfermo declarado sano) puede
costar una vida; un falso positivo solo genera estudios de más.
Métricas en orden de prioridad: sensibilidad (recall) como métrica rectora, AUC-ROC, especificidad, F1 y la
matriz de confusión. También conviene revisar la calibración de las probabilidades.

¿La medida de desempeño está alineada con el objetivo?
Sí, siempre que la sensibilidad se priorice sobre la exactitud global — el objetivo es no dejar pasar enfermos, y eso es
justo lo que mide.
Las clases están relativamente balanceadas (≈52% / 48%), así que la exactitud no engaña tanto. Aun así la asimetría
de costos persiste, por lo que el umbral de decisión no debe quedarse en 0.5: hay que bajarlo para favorecer la
sensibilidad.

¿Cuál sería el desempeño mínimo necesario?
Sensibilidad ≥ 0.85, AUC-ROC ≥ 0.80 y especificidad ≥ 0.70. La línea base ingenua (predecir siempre la clase
mayoritaria) da ≈52% de exactitud y 0% de sensibilidad; la literatura sobre este dataset reporta entre 80% y 88% de
exactitud.
Estos umbrales son una propuesta técnica: en un proyecto real se negocian con el personal médico.

¿Cuáles son los problemas parecidos? ¿Se puede reutilizar experiencia o herramientas?
Es clasificación binaria sobre datos tabulares mixtos, una de las familias más estudiadas. Equivale estructuralmente a
predicción de diabetes o ACV, detección de fraude o scoring crediticio.
Se reutiliza todo el instrumental estándar: scikit-learn, regresión logística como línea base interpretable, Random
Forest / Gradient Boosting para mejorar desempeño, y SHAP para explicar predicciones individuales — casi
obligatorio en aplicaciones clínicas.

¿Hay experiencia del problema disponible?
Abundante. Es uno de los datasets más usados en investigación desde 1988, con cientos de publicaciones como
referencia de desempeño. Además hay décadas de conocimiento clínico sobre estos factores de riesgo (estudios de
Framingham desde 1948) y el archivo datos_corazon_Info.txt describe cada variable.

¿Cómo se puede resolver el problema manualmente?
Un médico integra las señales que ya están en los datos:
1. Perfil demográfico: hombre >45 años o mujer >55 → riesgo basal alto
2. Tipo de dolor torácico: la angina típica orienta fuertemente a enfermedad coronaria
3. Angina de esfuerzo ( exang ) y depresión del ST ( old_peak ): los hallazgos más sugestivos de isquemia
4. max_hr baja para la edad: limitación funcional
5. Vasos comprometidos ( ca ): a más vasos, mayor certeza
6. Prueba de talio ( thal ): defecto reversible indica isquemia activa
Una regla manual como “angina de esfuerzo + depresión del ST + al menos un vaso comprometido → sospechar
enfermedad” ya rinde bastante. Sirve como línea base contra la cual comparar el modelo, y señala qué variables
deberían pesar más (verificable después con SHAP).
Listado de supuestos hasta este momento
Sobre los datos y el problema:
1. Las etiquetas se asignaron por confirmación diagnóstica, no por sospecha clínica
2. Las mediciones siguieron protocolos consistentes
3. Cada paciente aparece una sola vez
4. La muestra es representativa de la población objetivo
5. Las 13 variables contienen señal suficiente: no falta un predictor determinante
6. La solución opera como apoyo, nunca como diagnóstico autónomo
Sobre la calidad del archivo — verificado:
7. De 3.030 filas, 2.462 son duplicados exactos: solo 568 registros únicos
8. Todas las columnas tienen nulos, entre 30 ( age ) y 193 ( rest_ecg )
9. Hay valores basura inyectados: sex trae 2345 , 765 ; disease trae fsg , gsfdg ; thal trae 87654 , 56 ; exang
trae f , adfs . Por eso pandas lee casi todo como texto
10. Descartada la basura, los rangos son clínicamente plausibles: edad 29–77, presión 94–200 mmHg, colesterol
126–564 mg/dl
Es una versión deliberadamente ensuciada del original. Se asume que al limpiarla se recuperan los ~303 registros de
Cleveland.

¿Cuál es la fuente de los datos?
El Heart Disease Dataset del UCI Machine Learning Repository, base de Cleveland. Recolectado en la Cleveland
Clinic Foundation por el Dr. Robert Detrano y donado en 1988. El conjunto completo reúne cuatro bases con 76
atributos, pero casi toda la investigación usa la de Cleveland con 14 atributos y 303 registros — que es la de este
archivo.

¿Cómo se actualizan los datos?
No se actualizan: es un conjunto estático congelado desde 1988. En un despliegue real los datos vendrían de la
historia clínica electrónica, con la etiqueta confirmada después por angiografía.

¿Cada cuánto tiempo se actualizan los datos?
Nunca, en este archivo. Para un sistema en producción: ingesta diaria, reentrenamiento trimestral o semestral (o
cuando el monitoreo detecte deterioro) y revisión anual de supuestos con el equipo médico.
Una limitación de fondo: el dataset tiene casi 40 años. Los criterios diagnósticos y los perfiles poblacionales han
cambiado, así que un modelo entrenado con estos datos
